<a href="https://colab.research.google.com/github/williamfaraday123/SC4001-Neural-Network/blob/main/Lim_Isaac_Part_B_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question B4 (10 marks)

Model degradation is a common issue faced when deploying machine learning models (including neural networks) in the real world. New data points could exhibit a different pattern from older data points due to factors such as changes in government policy or market sentiments. For instance, housing prices in Singapore have been increasing and the Singapore government has introduced 3 rounds of cooling measures over the past years (16 December 2021, 30 September 2022, 27 April 2023).

In such situations, the distribution of the new data points could differ from the original data distribution which the models were trained on. Recall that machine learning models often work with the assumption that the test distribution should be similar to train distribution. When this assumption is violated, model performance will be adversely impacted.  In the last part of this assignment, we will investigate to what extent model degradation has occurred.




---



---



Your co-investigators used a linear regression model to rapidly test out several combinations of train/test splits and shared with you their findings in a brief report attached in Appendix A below. You wish to investigate whether your deep learning model corroborates with their findings.

In [ ]:
!pip install alibi-detect

In [ ]:
!pip install "pytorch_tabular[extra]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 14.4 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.0
    Uninstalling wandb-0.25.0:
      Successfully uninstalled wandb-0.25.0


In [ ]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from alibi_detect.cd import TabularDrift

import torch
import omegaconf
import collections
import typing

# The "Master List" for PyTorch 2.6+ security limits
torch.serialization.add_safe_globals([
    # Python Primitives and Collections
    dict, list, set, str, int, float, bool,
    collections.defaultdict,
    collections.OrderedDict,

    # Typing modules used in metadata
    typing.Any, typing.Union, typing.Optional, typing.List, typing.Dict,

    # OmegaConf (The config library)
    omegaconf.dictconfig.DictConfig,
    omegaconf.listconfig.ListConfig,
    omegaconf.nodes.AnyNode,
    omegaconf.base.ContainerMetadata,
    omegaconf.base.Metadata,

    # Numpy types often found in state dicts
    np.dtype,
    # Depending on your numpy version, this might be needed:
    # np._core.multiarray.scalar
])


1.Evaluate your model from B1 on data from year 2022 and report the test R2.

In [ ]:
df = pd.read_csv('hdb_price_prediction.csv')

# TODO: Enter your code here
import pytorch_tabular
from sklearn.metrics import r2_score

from google.colab import drive
drive.mount('/content/drive')

model = pytorch_tabular.tabular_model.TabularModel.load_model("/content/drive/MyDrive/hdb_models/b1")

test_data = df[df['year'] == 2022]
predictions = model.predict(test_data)

print("R2: ", r2_score(test_data['resale_price'], predictions['resale_price_prediction']))

Mounted at /content/drive


/usr/lib/python3.12/pickle.py:1557: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  obj = cls.__new__(cls, *args)
2026-03-12 03:14:01,923 - {pytorch_tabular.tabular_model:170} - INFO - Experiment Tracking is turned off
2026-03-12 03:14:01,927 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and ar

R2:  -10.423493591270804


2.Evaluate your model from B1 on data from year 2023 and report the test R2.

In [ ]:
# TODO: Enter your code here
test_data = df[df['year'] == 2023]
predictions = model.predict(test_data)

print("R2: ", r2_score(test_data['resale_price'], predictions['resale_price_prediction']))

R2:  -10.95143503413201


3.Did model degradation occur for the deep learning model?


In [ ]:
# YOUR ANSWER HERE
"""
Yes, model degradation occurred for this model,
as the model is getting worse over time due to technical rot,
as shown by the Test R2 values decreasing from -9.8873 in 2021 (from Part_B_1)
to -10.423493591270804 in 2022 to -10.95143503413201 in 2023.

In this specific case, it is due to Temporal Data Drift. My model was trained on 2017–2019 data. During that period, HDB prices were relatively stable.
Starting in late 2020 and peaking through 2022–2023, the Singapore property market experienced a historic upward shift in prices due to pandemic-related supply constraints and increased demand.
"""



---



---



4.Model degradation could be caused by [various data distribution shifts](https://huyenchip.com/2022/02/07/data-distribution-shifts-and-monitoring.html#data-shift-types): covariate shift (features), label shift and/or concept drift (altered relationship between features and labels).
There are various conflicting terminologies in the [literature](https://www.sciencedirect.com/science/article/pii/S0950705122002854#tbl1). Let’s stick to this reference for this assignment.

> Using the **Alibi Detect** library, apply the **TabularDrift** function with the training data (year 2019 and before) used as the reference and **detect which features have drifted** in the 2023 test dataset. Before running the statistical tests, ensure you **sample 800 data points** each from the train and test data. Do not use the whole train/test data. (Hint: use this example as a guide https://docs.seldon.io/projects/alibi-detect/en/stable/examples/cd_chi2ks_adult.html)


In [ ]:
# YOUR CODE HERE

target = ["resale_price"]
categorical_cols = ["month", "town", "flat_model_type", "storey_range"]
continuous_cols = [
    "dist_to_nearest_stn",
    "dist_to_dhoby",
    "degree_centrality",
    "eigenvector_centrality",
    "remaining_lease_years",
    "floor_area_sqm",
]

# Extract unique categories for each categorical column
category_map = {}
for i, col in enumerate(categorical_cols):
    category_map[i] = df[col].unique().tolist()

n_train = 800
n_test = 800

X_train = df[df["year"] <= 2019]
X_test = df[df["year"] == 2023]

X_train = X_train[:n_train] # Sample from the dataset
X_test = X_test[:n_test] # Sample from the dataset

X_ref = X_train[categorical_cols + continuous_cols].values
X_test = X_test[categorical_cols + continuous_cols].values

y_ref = X_train[target].values
y_test = df[df["year"] == 2023][target].values[:n_test]

categories_per_feature = {f: None for f in list(category_map.keys())}
cd = TabularDrift(
    X_ref, p_val=0.05, categories_per_feature=categories_per_feature
)

predictions = cd.predict(X_test)
labels = ['No','Yes']
print('Drift? {}'.format(labels[predictions['data']['is_drift']]))

fpreds = cd.predict(X_test, drift_type='feature')
feature_names = categorical_cols + continuous_cols

for f in range(cd.n_features):
    stat = 'Chi2' if f in list(categories_per_feature.keys()) else 'K-S'
    fname = feature_names[f]
    is_drift = fpreds['data']['is_drift'][f]
    stat_val, p_val = fpreds['data']['distance'][f], fpreds['data']['p_val'][f]
    print(f'{fname} -- Drift? {labels[is_drift]} -- {stat} {stat_val:.3f} -- p-value {p_val:.3f}')

Drift? Yes
month -- Drift? No -- Chi2 0.000 -- p-value 1.000
town -- Drift? Yes -- Chi2 537.272 -- p-value 0.000
flat_model_type -- Drift? Yes -- Chi2 51.600 -- p-value 0.016
storey_range -- Drift? Yes -- Chi2 38.958 -- p-value 0.001
dist_to_nearest_stn -- Drift? No -- K-S 0.062 -- p-value 0.084
dist_to_dhoby -- Drift? Yes -- K-S 0.159 -- p-value 0.000
degree_centrality -- Drift? No -- K-S 0.014 -- p-value 1.000
eigenvector_centrality -- Drift? Yes -- K-S 0.149 -- p-value 0.000
remaining_lease_years -- Drift? Yes -- K-S 0.239 -- p-value 0.000
floor_area_sqm -- Drift? Yes -- K-S 0.126 -- p-value 0.000


5.Assuming that the flurry of housing measures have made an impact on the relationship between all the features and resale_price (i.e. P(Y|X) changes), which type of data distribution shift possibly led to model degradation?


In [ ]:
# YOUR ANSWER HERE
"""
Concept Drift. If the housing measures change the relationship between all the features and the resale_price, this would impact the conditions by which X impacts Y, thereby changing P(Y|X). Hence, despite the fact that our input features selected (X) for the model did not change, changes in the 'conditions' of the housing market still impact the resale price (Y) of the house. Thereby reflecting a case of Concept Drift which possibly led to model degradation.
"""

6.From your analysis via TabularDrift, which features contribute to this shift?


In [ ]:
# YOUR ANSWER HERE
[
    "town",
    "flat_model_type",
    "storey_range",
    "dist_to_dhoby",
    "eigenvector_centrality",
    "remaining_lease_years",
    "floor_area_sqm"
]

7.Suggest 1 way to address model degradation and implement it, showing improved test R2 for year 2023.


In [ ]:
# YOUR CODE HERE
# We can try to train the model on the reference year in appendix A where the
# data is likely representative of the data in 2023.
model.fit(train=df[df['year'] == 2022])
test_data = df[df['year'] == 2023]

predictions = model.predict(test_data)

print("R2: ", r2_score(test_data['resale_price'], predictions['resale_price_prediction']))

INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-03-12 05:24:11,841 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-12 05:24:11,868 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-12 05:24:11,956 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-12 05:24:12,003 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-03-12 05:24:12,054 - {pytorch_tabular.tabular_model:655} - INFO - Auto LR Find Sta

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/.lr_find_ff27c8d1-c5db-4499-bd2f-54d76c34d4eb.ckpt
INFO:pytorch_lightning.utilities.rank_zero:Restored all states from the checkpoint at /content/.lr_find_ff27c8d1-c5db-4499-bd2f-54d76c34d4eb.ckpt
INFO:pytorch_lightning.tuner.lr_finder:Learning rate set to 5.7543993733715664e-05
2026-03-12 05:24:15,612 - {pytorch_tabular.tabular_model:668} - INFO - Suggested LR: 5.7543993733715664e-05. For plot and detailed analysis, use `find_learning_rate` method.
2026-03-12 05:24:15,614 - {pytorch_tabular.tabular_model:677} - INFO - Training Started


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  3.0 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.6 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.6 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.6 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


2026-03-12 05:25:25,248 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-12 05:25:25,249 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


R2:  -10.95149680554047


### Appendix A



Here are our results from a linear regression model. We used StandardScaler for continuous variables and OneHotEncoder for categorical variables.

While 2021 data can be predicted well, test R2 dropped rapidly for 2022 and 2023.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| Year <= 2020 | 2021     | 0.76    |
| Year <= 2020 | **2022**     | 0.41    |
| Year <= 2020 | **2023**     | **0.10**   |



Similarly, a model trained on 2017 data can predict 2018-2021 well (with slight degradation in performance for 2021), but drops drastically in 2022 and 2023.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| 2017         | 2018     | 0.90    |
|              | 2019     | 0.89    |
|              | 2020     | 0.87    |
|              | 2021     | 0.72    |
|              | **2022**     | **0.37**    |
|              | **2023**     | **0.09**    |

With the test set fixed at year 2021, training on data from 2017-2020 still works well on the test data, with minimal degradation. Training sets closer to year 2021 generally do better.

| Training set | Test set | Test R2 |
|--------------|----------|---------|
| 2020         | 2021     | 0.81    |
| 2019         | 2021     | 0.75    |
| 2018         | 2021     | 0.73    |
| 2017         | 2021     | 0.72    |